# Session 0c - NumPy Essentials

**Asynchronous · ~90 minutes · Required before Session 8**

---

## Learning objectives

By the end of this notebook you will be able to:

1. State the shape of any array you create, before running the cell.
2. Use `axis` correctly, and explain what it collapses.
3. Predict whether two arrays will broadcast, and to what shape.
4. Distinguish `*` from `@`, and say which one a neural network needs where.
5. Read a shape-mismatch error and know which operand to fix.

## Why this notebook exists

Sessions 1–7 use pandas, and you can get a long way in pandas without thinking about
array shapes. **Session 8 is different.** You will implement a neural network in
NumPy, and essentially every bug you hit will be a shape bug.

This notebook is not a NumPy tour. It covers the six things Session 8 actually
requires, and it drills the one habit that prevents most of the pain:

> **Annotate every array with its shape, in a comment, as you write it.**

Practitioners who do this write working array code. Practitioners who do not spend
their evenings reading `ValueError: matmul: Input operand 1 has a mismatch`.

In [ ]:
import numpy as np

print("numpy", np.__version__)
np.set_printoptions(precision=4, suppress=True)

## §1 - Arrays have shapes, and the shape is the thing to track

A NumPy array is a grid of numbers plus a **shape**: a tuple saying how long it is
along each axis.

In [ ]:
scalar = np.array(3.0)                       # shape ()      - zero axes
vector = np.array([1.0, 2.0, 3.0])           # shape (3,)    - one axis
matrix = np.array([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])         # shape (2, 3)  - two axes

for name, arr in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(f"  {name:8s} shape {str(arr.shape):8s} ndim {arr.ndim}  size {arr.size}")

### The distinction that causes the most trouble

`(3,)` and `(3, 1)` and `(1, 3)` all contain three numbers and are **not
interchangeable**.

| shape | what it is | reads as |
|---|---|---|
| `(3,)` | a 1-D array | "three numbers", no orientation |
| `(3, 1)` | a 2-D array | a **column**: 3 rows, 1 column |
| `(1, 3)` | a 2-D array | a **row**: 1 row, 3 columns |

In [ ]:
flat = np.array([1.0, 2.0, 3.0])
column = flat.reshape(-1, 1)     # -1 means "work it out from the total size"
row = flat.reshape(1, -1)

print(f"  flat   {flat.shape}\n{flat}\n")
print(f"  column {column.shape}\n{column}\n")
print(f"  row    {row.shape}\n{row}")

`reshape(-1, 1)` is the idiom you will use constantly in Session 8, because scikit-learn
hands you a target of shape `(n,)` and a network's output layer produces `(n, 1)`.
Getting those to line up is a recurring small task.

In [ ]:
# TODO: Before running, write down the shape each line produces. Then check.
#
a = np.zeros(4)                  # shape: ____
b = np.ones((2, 5))              # shape: ____
c = np.arange(6).reshape(3, 2)   # shape: ____
d = c.T                          # shape: ____
e = np.arange(6).reshape(-1, 3)  # shape: ____

# print them and compare with your predictions

## §2 - Element-wise operations

Arithmetic between arrays of the same shape works element by element.

In [ ]:
p = np.array([1.0, 2.0, 3.0])
q = np.array([10.0, 20.0, 30.0])

print(f"  p + q  = {p + q}")
print(f"  p * q  = {p * q}      <- element-wise, NOT a dot product")
print(f"  p ** 2 = {p ** 2}")
print(f"  p > 2  = {p > 2}       <- a boolean array, same shape")

`*` is **element-wise multiplication**. This is worth stating loudly because in
Session 8 you need both `*` and `@`, in different places, and confusing them
produces either an error or - worse - a plausible wrong answer.

| operator | name | Session 8 use |
|---|---|---|
| `*` | element-wise (Hadamard) | applying the ReLU derivative mask: `dA1 * (Z1 > 0)` |
| `@` | matrix multiplication | the affine step: `X @ W1` |

## §3 - `axis`: the one that everybody gets wrong

Reductions like `sum` and `mean` take an `axis` argument. The rule, stated the way
that actually helps:

> **`axis=k` collapses axis k.** Whatever length that axis had, it disappears from
> the output shape.

For a `(2, 3)` matrix - 2 rows, 3 columns:

- `axis=0` collapses the **rows**, leaving one value **per column** → shape `(3,)`
- `axis=1` collapses the **columns**, leaving one value **per row** → shape `(2,)`

Most people memorise "axis=0 is columns" and then get confused, because the number
0 refers to the axis being *removed*, not the thing being reported.

In [ ]:
M = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])          # (2, 3)

print(f"  M.sum()        = {M.sum()}          shape {np.shape(M.sum())}  everything")
print(f"  M.sum(axis=0)  = {M.sum(axis=0)}  shape {M.sum(axis=0).shape}  per column")
print(f"  M.sum(axis=1)  = {M.sum(axis=1)}       shape {M.sum(axis=1).shape}  per row")
print()
print("  keepdims=True preserves the axis as length 1, which keeps broadcasting easy:")
print(f"  M.sum(axis=1, keepdims=True) shape {M.sum(axis=1, keepdims=True).shape}")

### Why this matters in Session 8

Your data matrix is `(n_examples, n_features)`. So:

- **`axis=0`** collapses examples → a per-feature quantity. This is what you want for
  summing gradients over a batch: `db1 = dZ1.sum(axis=0)`.
- **`axis=1`** collapses features → a per-example quantity. This is what you want for
  a per-example loss.

Use the wrong one and you get a gradient of the wrong shape - if you are lucky, an
error; if not, a silently broadcast result and a network that trains to nonsense.

In [ ]:
# TODO: `data` has shape (4, 3): four examples, three features.
data = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0],
                 [7.0, 8.0, 9.0],
                 [10.0, 11.0, 12.0]])

# Write the ONE line for each. Predict the shape first.
#
# 1. The mean of each FEATURE (three numbers).
feature_means = ...          # predicted shape: ____

# 2. The sum of each EXAMPLE (four numbers).
example_sums = ...           # predicted shape: ____

# 3. `data` with each feature centred (its own mean subtracted). Shape (4, 3).
centred = ...                # predicted shape: ____

## §4 - Broadcasting

NumPy will operate on arrays of *different* shapes by stretching the smaller one,
if the shapes are compatible. This is what made `data - data.mean(axis=0)` work: a
`(4, 3)` minus a `(3,)`.

### The rule

Align shapes **from the right**. For each position, the lengths must be equal, or one
of them must be 1 (a missing axis counts as 1).

```
  (4, 3)          (2, 3)          (3,)            (3, 1)
  (   3)          (2, 1)          (3, 1)          (1, 4)
  ------          ------          ------          ------
  (4, 3)  ok      (2, 3)  ok      (3, 3)  !!      (3, 4)  !!
```

The last two are the dangerous ones. They **succeed**, and they produce something
much larger than you intended.

In [ ]:
print("  the safe, intended case:")
print(f"    (2,3) + (3,)   -> {(np.ones((2, 3)) + np.ones(3)).shape}")
print(f"    (2,3) + (2,1)  -> {(np.ones((2, 3)) + np.ones((2, 1))).shape}")
print()
print("  the trap:")
u = np.array([1.0, 2.0, 3.0])              # (3,)
v = np.array([[10.0], [20.0], [30.0]])     # (3, 1)
print(f"    u.shape = {u.shape},  v.shape = {v.shape}")
print(f"    u + v   -> shape {(u + v).shape}   <- a 3x3 matrix, not 3 numbers")
print(u + v)

### Why this is the bug that hurts most

Nothing raised an error. You asked for three numbers and got nine, and if the next
step is a `.mean()` you will get a plausible-looking number that is wrong.

In Session 8 the concrete version is: your target `y` has shape `(n,)` and your
network output has shape `(n, 1)`. Compute `output - y` and you get an `(n, n)`
matrix of every prediction minus every target. Your loss will be a number, it will
even go down during training, and your network will learn nothing useful.

> **The defence:** `reshape(-1, 1)` your targets once, at the top, and annotate every
> array with its shape so a mismatch is visible in the source rather than in the
> output.

In [ ]:
# TODO: Predict each result: a shape, or "error". Then run.
#
#   np.ones((5, 3))  +  np.ones(3)          -> ____
#   np.ones((5, 3))  +  np.ones(5)          -> ____
#   np.ones((5, 3))  +  np.ones((5, 1))     -> ____
#   np.ones((5, 1))  +  np.ones(3)          -> ____
#
# The second and fourth are the instructive ones. Say why before you run it.

## §5 - Matrix multiplication

`@` is matrix multiplication. The rule: `(a, b) @ (b, c)` gives `(a, c)`. The inner
dimensions must match, and they **vanish**.

```
    (n, d) @ (d, h)  ->  (n, h)
             ^    ^
             these must match, and disappear
```

This single line is a neural network layer: `n` examples with `d` features each,
times a weight matrix mapping `d` inputs to `h` hidden units, giving `n` examples
with `h` hidden activations.

In [ ]:
n, d, h = 4, 3, 2
X = np.arange(n * d, dtype=float).reshape(n, d)      # (4, 3)
W = np.ones((d, h))                                  # (3, 2)
b = np.array([0.5, -0.5])                            # (2,)

Z = X @ W + b                                        # (4,2) + (2,) broadcasts
print(f"  X {X.shape} @ W {W.shape} = {(X @ W).shape}")
print(f"  + b {b.shape} broadcasts -> Z {Z.shape}")
print(Z)

print("\n  the mismatch, and how to read it:")
try:
    _ = X @ np.ones((h, d))          # (4,3) @ (2,3): inner dims 3 vs 2
except ValueError as exc:
    print(f"    {exc}")
print("    -> 'size 3 is different from 2': the INNER dimensions disagree.")
print("       Fix by transposing one operand, or by fixing the weight's shape.")

### The outer product

One more you need: `np.outer(u, v)` takes a `(m,)` and an `(n,)` and gives `(m, n)`.
It appears in Session 8 as the weight gradient - an outer product of the incoming
gradient and the layer's input.

In [ ]:
print(f"  np.outer([1,2], [10,20,30]) shape "
      f"{np.outer([1, 2], [10, 20, 30]).shape}")
print(np.outer([1, 2], [10, 20, 30]))

## §6 - Boolean masks

A comparison gives a boolean array of the same shape, which you can use to select or
to zero out.

In [ ]:
vals = np.array([-2.0, -0.5, 0.0, 1.5, 3.0])

print(f"  vals            = {vals}")
print(f"  vals > 0        = {vals > 0}")
print(f"  vals[vals > 0]  = {vals[vals > 0]}          <- selection, shape changes")
print(f"  vals * (vals>0) = {vals * (vals > 0)}   <- masking, shape preserved")
print(f"  np.maximum(0,v) = {np.maximum(0.0, vals)}   <- this is ReLU")

The third line is exactly how the ReLU derivative is applied in backpropagation:
`dZ1 = dA1 * (Z1 > 0)`. The boolean array is multiplied in as 1s and 0s, passing the
gradient where the unit was active and blocking it where it was not.

Note the difference between the two middle lines. **Selection** (`vals[mask]`) changes
the shape; **masking** (`vals * mask`) preserves it. Backpropagation needs the second,
because the gradient must keep the same shape as the thing it is a gradient of.

## §7 - Debugging exercise

Below is a function that is supposed to compute a two-layer forward pass and a
per-example squared loss, for `n=5` examples with `d=3` features and `h=2` hidden
units. It should return predictions of shape `(5, 1)`.

It raises this instead:

```
ValueError: operands could not be broadcast together with shapes (2,6) (5,)
```

Look at those two shapes. **Neither of them is a shape you asked for.** There is no
6 anywhere in the problem, and `(2,6)` is not the shape of anything the function was
supposed to produce.

That is the characteristic signature of an array bug: the error surfaces at the line
where the shapes finally become *incompatible*, which can be several steps after the
line where they first became *wrong*. The traceback points at the loss computation.
None of the three bugs is there.

Find all three. Then answer the question that matters: **which of them would have
been silent on its own?**

In [ ]:
def broken_forward(X, W1, b1, W2, b2, y):
    """Supposed to return (predictions, mean squared error).

    X:  (n, d)      W1: (d, h)     b1: (h,)
                    W2: (h, 1)     b2: (1,)
    y:  (n,)
    Should return predictions of shape (n, 1) and a scalar loss.
    """
    Z1 = X @ W1 + b1
    A1 = Z1[Z1 > 0]
    Z2 = A1 * W2 + b2
    loss = np.mean((Z2 - y) ** 2)
    return Z2, loss


# TODO: Find and fix the three bugs. For each, write down:
#   - what the shape should have been, and what it actually was
#   - whether that bug ALONE would have raised an error
#
# Work forwards from Z1, printing .shape after every line. The first shape that is
# not what you expected is the first bug - not where the traceback points.
#
# Test with:  n, d, h = 5, 3, 2